# GCD kernel spectra

Numerical companion to the positive-semidefiniteness of the normalized gcd kernel.

**Claim evidenced.** The kernel $K(m,n) = \gcd(m,n)/\sqrt{mn}$ is positive
semidefinite. This is *proved* in Lean as `normalizedGcdKernel_posSemidef`; this
notebook is only the numerical companion. We compute the spectra of the finite
sections $K_N = (K(m,n))_{1\le m,n\le N}$ for $N \in \{50, 200, 800, 2000\}$ and
check that every eigenvalue is nonnegative (up to floating-point noise
$\ge -10^{-10}$) and that the diagonal is identically $1$.

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np


def gcd_kernel(n: int) -> np.ndarray:
    """K_N with entries gcd(m, k) / sqrt(m k), 1 <= m, k <= n."""
    i = np.arange(1, n + 1)
    g = np.gcd.outer(i, i).astype(float)
    return g / np.sqrt(np.outer(i, i))


Ns = [50, 200, 800, 2000]
spectra = {N: np.linalg.eigvalsh(gcd_kernel(N)) for N in Ns}

In [2]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for N in Ns:
    vals = spectra[N]
    ax.semilogy(np.arange(1, N + 1) / N, vals[::-1], lw=1, label=f"N={N}")
ax.set_xlabel("normalized index $k/N$")
ax.set_ylabel("eigenvalue (log scale)")
ax.set_title("Spectrum of the normalized gcd kernel $K_N$")
ax.legend()
fig.tight_layout()
plt.show()

<Figure size 800x450 with 1 Axes>

In [3]:
print(f"{'N':>6} {'min eigenvalue':>18} {'max eigenvalue':>18}")
for N in Ns:
    print(f"{N:>6} {spectra[N][0]:>18.6e} {spectra[N][-1]:>18.6e}")

min_eigs = np.array([spectra[N][0] for N in Ns])
assert np.all(min_eigs >= -1e-10), "an eigenvalue dipped below the -1e-10 tolerance"
print("\nall minimum eigenvalues >= -1e-10:", bool(np.all(min_eigs >= -1e-10)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(Ns, min_eigs, "o-")
ax.set_xlabel("N")
ax.set_ylabel("smallest eigenvalue of $K_N$")
ax.set_title("Minimum eigenvalue stays strictly positive")
fig.tight_layout()
plt.show()

     N     min eigenvalue     max eigenvalue
    50       4.864186e-02       8.468936e+00
   200       2.929228e-02       1.384027e+01
   800       1.856600e-02       2.110508e+01
  2000       1.447072e-02       2.713237e+01

all minimum eigenvalues >= -1e-10: True


<Figure size 600x400 with 1 Axes>

In [4]:
# Unit diagonal: K(n, n) = gcd(n, n)/sqrt(n*n) = n/n = 1 exactly.
for N in Ns:
    diag_err = np.max(np.abs(np.diag(gcd_kernel(N)) - 1.0))
    print(f"N={N:>5}: max |K(n,n) - 1| = {diag_err:.3e}")

N=   50: max |K(n,n) - 1| = 0.000e+00
N=  200: max |K(n,n) - 1| = 0.000e+00
N=  800: max |K(n,n) - 1| = 0.000e+00
N= 2000: max |K(n,n) - 1| = 0.000e+00


**Interpretation.** Every finite section $K_N$ up to $N = 2000$ has a strictly
positive spectrum: the smallest eigenvalue decays with $N$ (the kernel is far
from uniformly coercive) but never crosses zero, staying comfortably above the
$-10^{-10}$ floating-point tolerance, and the diagonal is exactly $1$. This is
numerical evidence consistent with positive semidefiniteness — it checks finitely
many $N$ in floating point and proves nothing by itself; the actual proof is the
Lean theorem `normalizedGcdKernel_posSemidef`.